# 03 · Build the iNaturalist v2 benchmark

Builds the main benchmark from iNaturalist 2021 (train-mini) and extracts every cached feature the analyses need.
Run it once on a CUDA machine; all later analyses run from the cached features.

**Outputs** (all inside `files/v2/`; nothing existing is overwritten):

| Output | Purpose |
|---|---|
| `species_pool.json` | 2,017 species (up to 200 per supercategory) |
| `species_split_v2.json` | the benchmark split, with computed ground truth for every novel species |
| `features_{siglip2,dinov3,dinov2}_v2.npz` | frozen backbone features for every pool image |
| `features_trifusion_v2.npz` | trained fusion features (fusion trained on known species only) |
| `alt_splits/features_trifusion_<pair>.npz` | one fusion per hold-out pair, for the transfer test across five splits |
| `MANIFEST.json` | the files to copy to the analysis machine |

**Design principles**

1. **Enough novel species.** 180 placeable novel species, so placement accuracy has a usable confidence interval.
2. **Realistic hold-outs.** Whole *orders* are withheld inside each group, in addition to withholding two
   supercategories (Fungi, Reptiles) in full.
3. **Ground truth is computed, never assumed.** Each novel species' route (place vs. construct) comes from its
   true deepest ancestor in the known set. Reptiles, for example, correctly attach at phylum Chordata.
4. **Pool larger than the split.** Splits are only labels, so a redesign needs no new feature extraction.

The JSON files in `data/inat_v2/` of this repository are the canonical benchmark. The pool step draws the species
of an earlier 550-species split first when that file (`species_split.json`) is present, so an exact rebuild of the
pool needs it.

**Run order:** top to bottom. Every slow step is resumable: re-run the same cell and it continues where it stopped.


## Step 1 — Configuration

Paths follow the original pipeline. The only things you may need to edit are the candidate locations of
`train_mini.json` and `train_mini.tar.gz` if they live somewhere else on this machine.

In [ ]:
# CELL: setup
import json, os, sys, time, random, shutil, tarfile
from pathlib import Path
from collections import Counter, defaultdict
import numpy as np
import torch

# ---------------- paths: same machine and layout as the original pipeline ----------------
BASE_DIR  = Path(os.environ.get("HNSD_BASE_DIR", r"C:\hnsd\files"))   # data root; set HNSD_BASE_DIR or edit
IMG_ROOT  = BASE_DIR / "selected"             # images: selected/train_mini/<species folder>/*.jpg
OLD_SPLIT = BASE_DIR / "species_split.json"   # optional earlier 550-species split; only orders the pool (read only)
V2_DIR    = BASE_DIR / "v2"                   # EVERYTHING this notebook writes goes here
SHARD_DIR = V2_DIR / "feature_shards"

# iNaturalist 2021 mini files - the first existing path wins. Add yours if different.
INAT_JSON_CANDIDATES = [BASE_DIR / "train_mini.json",
                        BASE_DIR.parent / "train_mini.json"]
INAT_TAR_CANDIDATES  = [BASE_DIR / "train_mini.tar.gz",
                        BASE_DIR.parent / "train_mini.tar.gz"]

# ---------------- benchmark design ----------------
SEED                       = 42
POOL_PER_GROUP             = 200   # species extracted per supercategory (capped by availability)
HELD_OUT_SUPERCATS         = ["Fungi", "Reptiles"]   # supercategory-level condition, as before
UNSEEN_SUPERCAT_PER_GROUP  = 100   # of the pool, how many Fungi / Reptiles enter the split
ORDER_OUT_MIN_SPECIES      = 15    # per group: withhold whole orders until at least this many species
NOVEL_KNOWN_PER_GROUP      = 20    # per group: single species withheld with a known relative (9 x 20 = 180)
N_VAL_SPECIES              = 120   # known species held back to early-stop the fusion

# five-split hold-out pairs (the generality test of the transfer mechanism)
ALT_PAIRS = [["Fungi", "Reptiles"], ["Amphibians", "Mollusks"], ["Arachnids", "Ray-finned Fishes"],
             ["Birds", "Insects"], ["Mammals", "Plants"]]

# ---------------- backbones ----------------
SIGLIP2_ID = "google/siglip2-so400m-patch16-naflex"
DINOV3_ID  = "facebook/dinov3-vitl16-pretrain-lvd1689m"   # gated: uses your cached HF login
DINOV2_ID  = "facebook/dinov2-large"
HF_TOKEN   = None      # leave None to use the cached `huggingface-cli login`
BATCH      = 48        # halves automatically on out-of-memory
IO_WORKERS = 8         # threads decoding JPEGs
SHARD_SPECIES = 50     # species per resumable shard

RANKS  = ["kingdom", "phylum", "class", "order", "family", "genus"]
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

V2_DIR.mkdir(parents=True, exist_ok=True); SHARD_DIR.mkdir(parents=True, exist_ok=True)
free_gb = shutil.disk_usage(BASE_DIR).free / 1e9
print(f"device      : {DEVICE}" + (f" ({torch.cuda.get_device_name(0)})" if DEVICE == "cuda" else ""))
print(f"base dir    : {BASE_DIR}  exists={BASE_DIR.exists()}")
print(f"free disk   : {free_gb:.1f} GB  (new images ~6-8 GB, features ~2 GB)")
if free_gb < 15:
    print("WARNING: under 15 GB free. Lower POOL_PER_GROUP or free space before continuing.")

## Step 2 — Load the iNaturalist taxonomy

Reads `train_mini.json` (about 200 MB) once to get the full lineage of all 10,000 species and the image
files of each. If it is not on this machine, download it from the iNaturalist 2021 site first.

In [ ]:
# CELL: load_inat
def first_existing(paths):
    for p in paths:
        if Path(p).exists():
            return Path(p)
    return None

INAT_JSON = first_existing(INAT_JSON_CANDIDATES)
if INAT_JSON is None:
    raise FileNotFoundError(
        "train_mini.json not found. Download it from the iNaturalist 2021 site "
        "(github.com/visipedia/inat_comp) into " + str(BASE_DIR))

t0 = time.time()
with open(INAT_JSON) as f:
    _inat = json.load(f)
CATS = {c["id"]: c for c in _inat["categories"]}
_file_of_image = {im["id"]: im["file_name"] for im in _inat["images"]}
FILES_BY_CAT = defaultdict(list)
for a in _inat["annotations"]:
    FILES_BY_CAT[a["category_id"]].append(_file_of_image[a["image_id"]])
del _inat, _file_of_image

print(f"loaded {INAT_JSON.name} in {time.time()-t0:.0f}s")
print(f"species: {len(CATS)} | images: {sum(len(v) for v in FILES_BY_CAT.values())}")
print("supercategories:", dict(sorted(Counter(c["supercategory"] for c in CATS.values()).items())))

## Step 3 — Build the species pool

Up to `POOL_PER_GROUP` species per supercategory. The original 550 species are always included (their images
are already on disk), and the rest are filled by a seeded random draw. The pool is deliberately larger than
any one split: features are extracted for all of it, so later split changes never need the GPU again.

In [ ]:
# CELL: pool
def species_record(c):
    files = FILES_BY_CAT[c["id"]]
    rec = {"category_id": c["id"], "name": c["name"], "common_name": c.get("common_name", ""),
           "supercategory": c["supercategory"]}
    rec.update({r: c[r] for r in RANKS})
    rec["image_dir"]  = "/".join(files[0].split("/")[:-1])      # e.g. train_mini/02957_Animalia_..._viridis
    rec["num_images"] = len(files)
    return rec

old_ids = set()
if OLD_SPLIT.exists():
    old_ids = {s["category_id"] for s in json.load(open(OLD_SPLIT))}

by_group = defaultdict(list)
for c in CATS.values():
    by_group[c["supercategory"]].append(c)

rng = random.Random(SEED)
POOL = []
for g in sorted(by_group):
    cs   = sorted(by_group[g], key=lambda c: c["id"])
    keep = [c for c in cs if c["id"] in old_ids]
    rest = [c for c in cs if c["id"] not in old_ids]
    rng.shuffle(rest)
    n = min(POOL_PER_GROUP, len(cs))
    POOL += [species_record(c) for c in keep + rest[:max(0, n - len(keep))]]
POOL.sort(key=lambda s: s["category_id"])
POOL_BY_ID = {s["category_id"]: s for s in POOL}

json.dump(POOL, open(V2_DIR / "species_pool.json", "w"), indent=1)
print(f"{'supercategory':<20}{'pool':>6}{'of which original':>19}{'available':>11}")
for g in sorted(by_group):
    n_pool = sum(1 for s in POOL if s["supercategory"] == g)
    n_old  = sum(1 for s in POOL if s["supercategory"] == g and s["category_id"] in old_ids)
    print(f"{g:<20}{n_pool:>6}{n_old:>19}{len(by_group[g]):>11}")
print(f"\npool: {len(POOL)} species, {sum(s['num_images'] for s in POOL)} images -> v2/species_pool.json")

## Step 4 — Design the new split

Three hold-out mechanisms create novel species:

| Mechanism | What is withheld | Typical true attachment |
|---|---|---|
| `species` | single species whose order keeps at least one known member | genus, family or order |
| `order` | whole orders inside a group (never the group's largest order) | class or above |
| `supercategory` | Fungi and Reptiles in full | phylum (Reptiles) or nothing (Fungi) |

**The route is then computed, not assigned.** For every novel species we walk its lineage from kingdom down
and record the deepest rank whose lineage prefix exists among the known species. Lineage *prefixes* are
compared rather than bare names, so homonyms in different parents cannot produce false matches. Species
whose deepest known ancestor is order, family or genus are `novel_known_branch` (route **place**);
everything coarser is `novel_unseen_branch` (route **construct**). The role names are the ones every
downstream analysis expects.

`design_split()` is a plain function of the pool, so the same split can be regenerated on the analysis machine.

In [ ]:
# CELL: split
def prefix(s, rank):
    """Lineage tuple from kingdom down to `rank`, e.g. ('Animalia','Chordata','Aves')."""
    return tuple(s[r] for r in RANKS[:RANKS.index(rank) + 1])

def deepest_known(s, known_prefixes):
    rank, parent = "none", None
    for r in RANKS:
        if prefix(s, r) in known_prefixes[r]:
            rank, parent = r, s[r]
        else:
            break
    return rank, parent

def design_split(pool, seed=SEED):
    rng = random.Random(seed)
    groups = defaultdict(list)
    for s in pool:
        groups[s["supercategory"]].append(s)

    mech = {}                                   # category_id -> hold-out mechanism
    for g in sorted(groups):
        members = sorted(groups[g], key=lambda s: s["category_id"])

        if g in HELD_OUT_SUPERCATS:
            rng.shuffle(members)
            for i, s in enumerate(members):
                mech[s["category_id"]] = "supercategory" if i < UNSEEN_SUPERCAT_PER_GROUP else "pool_only"
            continue

        # (a) whole orders, never the group's largest, only orders with >= 2 pool species
        order_size = Counter(prefix(s, "order") for s in members)
        largest = order_size.most_common(1)[0][0]
        cands = sorted(o for o, n in order_size.items() if o != largest and n >= 2)
        rng.shuffle(cands)
        withheld, n_out = set(), 0
        for o in cands:
            if n_out >= ORDER_OUT_MIN_SPECIES:
                break
            withheld.add(o); n_out += order_size[o]
        for s in members:
            if prefix(s, "order") in withheld:
                mech[s["category_id"]] = "order"

        # (b) single species, keeping at least one known relative in the same order
        remaining = [s for s in members if s["category_id"] not in mech]
        rng.shuffle(remaining)
        left = Counter(prefix(s, "order") for s in remaining)
        picked = 0
        for s in remaining:
            if picked >= NOVEL_KNOWN_PER_GROUP:
                break
            o = prefix(s, "order")
            if left[o] >= 2:
                mech[s["category_id"]] = "species"
                left[o] -= 1; picked += 1
        for s in remaining:
            mech.setdefault(s["category_id"], "none")

    # ground truth against the known set
    known = [s for s in pool if mech[s["category_id"]] == "none"]
    known_prefixes = {r: {prefix(k, r) for k in known} for r in RANKS}
    out = []
    for s in pool:
        m = mech[s["category_id"]]
        rec = dict(s); rec["holdout"] = m
        if m == "none":
            rec.update(role="known", true_rank="species", true_parent=s["name"])
        elif m == "pool_only":
            rec.update(role="pool_only", true_rank=None, true_parent=None)
        else:
            r, p = deepest_known(s, known_prefixes)
            rec.update(true_rank=r, true_parent=p,
                       role="novel_known_branch" if r in ("order", "family", "genus") else "novel_unseen_branch")
        out.append(rec)
    return out

SPLIT = design_split(POOL)
ROLE  = {s["category_id"]: s["role"] for s in SPLIT}
json.dump(SPLIT, open(V2_DIR / "species_split_v2.json", "w"), indent=1)
json.dump(ALT_PAIRS, open(V2_DIR / "alt_splits.json", "w"), indent=1)

# integrity checks
for s in SPLIT:
    if s["holdout"] == "species":
        assert s["role"] == "novel_known_branch", f"species hold-out without a known relative: {s['name']}"

print("ROLES:", dict(Counter(s["role"] for s in SPLIT)))
print(f"\n{'hold-out':<15}" + "".join(f"{r:>9}" for r in ["genus", "family", "order", "class", "phylum", "kingdom", "none"]))
for m in ["species", "order", "supercategory"]:
    c = Counter(s["true_rank"] for s in SPLIT if s["holdout"] == m)
    print(f"{m:<15}" + "".join(f"{c.get(r, 0):>9}" for r in ["genus", "family", "order", "class", "phylum", "kingdom", "none"]))

print("\nsupercategory hold-outs, true deepest known ancestor:")
for g in HELD_OUT_SUPERCATS:
    c = Counter((s["true_rank"], s["true_parent"]) for s in SPLIT if s["supercategory"] == g and s["holdout"] == "supercategory")
    print(f"  {g:<10} {dict(c)}")

n_place = sum(1 for s in SPLIT if s["role"] == "novel_known_branch")
print(f"\nplaceable novel species: {n_place} (earlier 550-species split: 45)")
print("saved v2/species_split_v2.json and v2/alt_splits.json")

## Step 5 — Get the images onto this machine

Checks which pool species are missing images under `selected/`. If `train_mini.tar.gz` is on this machine,
it streams through the archive **once** and extracts only the missing files, stopping as soon as everything
needed is found. It never overwrites existing files.

If the archive is not here, the cell writes `v2/folders_to_extract.txt` and stops. Extract those folders on
any machine that has the archive, copy them into `selected/train_mini/`, and re-run this cell.

In [ ]:
# CELL: images
def jpgs(folder):
    return sorted(Path(folder).glob("*.jpg")) if Path(folder).exists() else []

missing = {}                                  # image_dir -> number of files still missing
for s in POOL:
    have = len(jpgs(IMG_ROOT / s["image_dir"]))
    if have < s["num_images"]:
        missing[s["image_dir"]] = s["num_images"] - have
n_missing = sum(missing.values())
print(f"species with missing images: {len(missing)} | files missing: {n_missing}")

if missing:
    TAR = first_existing(INAT_TAR_CANDIDATES)
    if TAR is None:
        (V2_DIR / "folders_to_extract.txt").write_text("\n".join(sorted(missing)) + "\n")
        raise RuntimeError(
            "train_mini.tar.gz is not on this machine. The folder list is in v2/folders_to_extract.txt: "
            "extract them on the analysis machine, copy into selected/train_mini/, then re-run this cell.")

    print(f"streaming {TAR} (one pass, stops early when done) ...")
    t0, extracted, scanned = time.time(), 0, 0
    with tarfile.open(TAR, mode="r|gz") as tar:
        for m in tar:
            scanned += 1
            if not m.isfile():
                continue
            name = m.name[2:] if m.name.startswith("./") else m.name
            folder = name.rpartition("/")[0]
            if folder not in missing or ".." in name or not name.startswith("train_mini/"):
                continue
            dest = IMG_ROOT / name
            if dest.exists():
                continue
            dest.parent.mkdir(parents=True, exist_ok=True)
            src = tar.extractfile(m)
            with open(dest, "wb") as out:
                shutil.copyfileobj(src, out)
            extracted += 1
            if extracted % 5000 == 0:
                print(f"  extracted {extracted}/{n_missing}  ({time.time()-t0:.0f}s, scanned {scanned})")
            if extracted >= n_missing:
                break
    print(f"extracted {extracted} files in {time.time()-t0:.0f}s")

still = [s["image_dir"] for s in POOL if len(jpgs(IMG_ROOT / s["image_dir"])) < s["num_images"]]
print("all pool images present." if not still else f"WARNING: {len(still)} species still incomplete, e.g. {still[:3]}")

## Step 6 — Fix the image order

One ordered list of `(image path, species id)` for the whole pool, sorted by species id and then file name.
All three backbones encode exactly this order, so their features align row for row, and the relative path of
every image is stored with its features.

In [ ]:
# CELL: items
ITEMS = []
for s in POOL:                                        # POOL is sorted by category_id
    for p in jpgs(IMG_ROOT / s["image_dir"]):
        ITEMS.append((p.relative_to(IMG_ROOT).as_posix(), s["category_id"]))

species_order = [s["category_id"] for s in POOL]
SHARDS = []
for i in range(0, len(species_order), SHARD_SPECIES):
    chunk = set(species_order[i:i + SHARD_SPECIES])
    SHARDS.append([it for it in ITEMS if it[1] in chunk])

json.dump(ITEMS, open(V2_DIR / "items.json", "w"))
print(f"images: {len(ITEMS)} | shards: {len(SHARDS)} of {SHARD_SPECIES} species")

## Step 7 — Feature extractors

Preprocessing and pooling: SigLIP 2 image features via `get_image_features`, DINO features via `pooler_output`
(CLS token fallback), all L2-normalised. JPEG decoding runs on threads rather than worker processes, which avoids
the Windows pickling problem of DataLoader workers inside Jupyter.


In [ ]:
# CELL: extract_defs
from concurrent.futures import ThreadPoolExecutor
from PIL import Image

BACKBONES = {"siglip2": SIGLIP2_ID, "dinov3": DINOV3_ID, "dinov2": DINOV2_ID}

def load_backbone(name):
    from transformers import AutoModel, AutoProcessor, AutoImageProcessor
    mid = BACKBONES[name]
    if name == "siglip2":
        proc  = AutoProcessor.from_pretrained(mid)
        model = AutoModel.from_pretrained(mid)
    else:
        tok = (HF_TOKEN or True) if name == "dinov3" else None
        proc  = AutoImageProcessor.from_pretrained(mid, token=tok)
        model = AutoModel.from_pretrained(mid, token=tok)
    model = model.to(DEVICE).eval()
    for p in model.parameters():
        p.requires_grad = False
    return model, proc

@torch.no_grad()
def embed_images(name, model, proc, imgs):
    inp = proc(images=imgs, return_tensors="pt").to(DEVICE)
    if name == "siglip2":
        out = model.get_image_features(**inp)
        emb = out if torch.is_tensor(out) else out.pooler_output
    else:
        out = model(**inp)
        emb = out.pooler_output if getattr(out, "pooler_output", None) is not None else out.last_hidden_state[:, 0]
    emb = torch.nn.functional.normalize(emb.float(), dim=-1)
    return emb.cpu().numpy().astype(np.float32)

def load_rgb(rel):
    with Image.open(IMG_ROOT / rel) as im:
        return im.convert("RGB")

def shard_path(name, i):
    return SHARD_DIR / name / f"shard_{i:04d}.npz"

def run_backbone(name, batch=BATCH):
    (SHARD_DIR / name).mkdir(parents=True, exist_ok=True)
    todo = [i for i in range(len(SHARDS)) if not shard_path(name, i).exists()]
    if not todo:
        print(f"[{name}] all {len(SHARDS)} shards already done"); return
    print(f"[{name}] {len(todo)} of {len(SHARDS)} shards to do")
    model, proc = load_backbone(name)
    io = ThreadPoolExecutor(max_workers=IO_WORKERS)
    t0, n_img = time.time(), 0
    try:
        for k, i in enumerate(todo):
            items, feats, j = SHARDS[i], [], 0
            while j < len(items):
                chunk = items[j:j + batch]
                imgs = list(io.map(load_rgb, [r for r, _ in chunk]))
                try:
                    feats.append(embed_images(name, model, proc, imgs))
                    j += len(chunk)
                except torch.cuda.OutOfMemoryError:
                    torch.cuda.empty_cache()
                    batch = max(1, batch // 2)
                    print(f"  out of memory -> batch {batch}")
            tmp = shard_path(name, i).with_suffix(".tmp.npz")
            np.savez(tmp, X=np.concatenate(feats),
                     cat_ids=np.array([c for _, c in items], dtype=np.int64),
                     relpaths=np.array([r for r, _ in items]))
            os.replace(tmp, shard_path(name, i))     # atomic: a half-written shard never counts as done
            n_img += len(items)
            rate = n_img / max(time.time() - t0, 1e-6)
            left = sum(len(SHARDS[x]) for x in todo[k + 1:]) / max(rate, 1e-6)
            print(f"  shard {i+1}/{len(SHARDS)}  {rate:.0f} img/s  ~{left/60:.0f} min left for {name}")
    finally:
        io.shutdown()
        del model
        if DEVICE == "cuda":
            torch.cuda.empty_cache()

print("extractors ready:", list(BACKBONES))

## Step 8 — Throughput test (optional, recommended)

Times one backbone on 512 images and estimates the full run, so you know before committing the session.

In [ ]:
# CELL: throughput
TEST_N = min(512, len(ITEMS))
model, proc = load_backbone("siglip2")
rels = [r for r, _ in ITEMS[:TEST_N]]
b = BATCH
with ThreadPoolExecutor(max_workers=IO_WORKERS) as io:
    embed_images("siglip2", model, proc, list(io.map(load_rgb, rels[:8])))      # warm-up
    t0, j = time.time(), 0
    while j < TEST_N:
        try:
            embed_images("siglip2", model, proc, list(io.map(load_rgb, rels[j:j + b])))
            j += b
        except torch.cuda.OutOfMemoryError:
            if DEVICE == "cuda":
                torch.cuda.empty_cache()
            b = max(1, b // 2)
            print(f"out of memory -> batch {b}")
    dt = time.time() - t0
BATCH = b                                    # the extraction run starts from a batch size that fits
del model
if DEVICE == "cuda":
    torch.cuda.empty_cache()
ips = TEST_N / dt
print(f"SigLIP 2: {ips:.0f} images/s at batch {BATCH}")
print(f"rough estimate for all three backbones on {len(ITEMS)} images: ~{3*len(ITEMS)/ips/60:.0f} min")
print("(DINO models are usually a little faster than SigLIP 2, so this tends to overestimate.)")

## Step 9 — Extract features (the long step)

Resumable. If interrupted, run this cell again: finished shards are skipped, and a shard is only marked done
after it has been written completely.

In [ ]:
# CELL: extract_run
print(f"starting batch size: {BATCH}")
for name in BACKBONES:
    run_backbone(name, batch=BATCH)       # BATCH comes from the throughput test if it ran
print("all backbones done")

## Step 10 — Merge shards and check alignment

In [ ]:
# CELL: merge
def merge(name):
    parts = [np.load(shard_path(name, i)) for i in range(len(SHARDS))]
    X   = np.concatenate([p["X"] for p in parts])
    cid = np.concatenate([p["cat_ids"] for p in parts])
    rel = np.concatenate([p["relpaths"] for p in parts])
    np.savez(V2_DIR / f"features_{name}_v2.npz", X=X, cat_ids=cid, relpaths=rel)
    return X, cid, rel

FEATS = {}
for name in BACKBONES:
    FEATS[name] = merge(name)
    print(f"{name:<8} {FEATS[name][0].shape}")

ref_cid, ref_rel = FEATS["siglip2"][1], FEATS["siglip2"][2]
for name in BACKBONES:
    assert np.array_equal(FEATS[name][1], ref_cid), f"{name}: species order differs"
    assert np.array_equal(FEATS[name][2], ref_rel), f"{name}: image order differs"
    norms = np.linalg.norm(FEATS[name][0], axis=1)
    assert np.allclose(norms, 1, atol=1e-3), f"{name}: features not normalised"
assert len(ref_rel) == len(ITEMS)
CAT, REL = ref_cid, ref_rel
print("aligned: all three backbones share the same", len(CAT), "rows")

## Step 11 — Fusion module

A skip-anchored cross-attention fusion of the three encoders (d_model 256, dropout 0.4, gated residual with
alpha initialised at 0.1, AdamW lr 5e-5, weight decay 1e-2, supervised contrastive loss at temperature 0.1),
wrapped in one function so the main split and the five hold-out splits train identically. Training is
resumable per run.


In [ ]:
# CELL: fusion_defs
import torch.nn as nn
import torch.nn.functional as F
from sklearn.cluster import KMeans
from sklearn.metrics import normalized_mutual_info_score

class TriFusion(nn.Module):
    def __init__(self, d_s2, d_d3, d_d2, d_model=256, n_heads=4, out_dim=512, p=0.4):
        super().__init__()
        self.p_s2 = nn.Linear(d_s2, d_model); self.p_d3 = nn.Linear(d_d3, d_model); self.p_d2 = nn.Linear(d_d2, d_model)
        self.attn = nn.MultiheadAttention(d_model, n_heads, dropout=p, batch_first=True)
        self.norm = nn.LayerNorm(d_model); self.drop = nn.Dropout(p)
        self.mlp  = nn.Sequential(nn.Linear(3 * d_model, d_model), nn.GELU(), nn.Dropout(p), nn.Linear(d_model, out_dim))
        self.skip = nn.Linear(d_s2 + d_d3 + d_d2, out_dim)
        self.alpha = nn.Parameter(torch.tensor(0.1))
    def forward(self, a, b, c):
        t = torch.cat([self.p_s2(a).unsqueeze(1), self.p_d3(b).unsqueeze(1), self.p_d2(c).unsqueeze(1)], dim=1)
        ctx, _ = self.attn(t, t, t)
        ctx = self.norm(t + self.drop(ctx))
        adjust = self.mlp(ctx.reshape(ctx.size(0), -1))
        anchor = self.skip(torch.cat([a, b, c], dim=-1))
        return F.normalize(anchor + self.alpha * adjust, dim=-1)

def supcon_loss(z, labels, temp=0.1):
    sim = z @ z.T / temp
    eye = torch.eye(z.size(0), dtype=torch.bool, device=z.device)
    sim = sim.masked_fill(eye, -9e15)
    same = (labels[:, None] == labels[None, :]) & ~eye
    logp = sim - torch.logsumexp(sim, dim=1, keepdim=True)
    pc = same.sum(1); v = pc > 0
    if not v.any():
        return z.sum() * 0.0
    return (-(logp * same).sum(1)[v] / pc[v].clamp(min=1)).mean()

XS_T = torch.tensor(FEATS["siglip2"][0], device=DEVICE)
XD3_T = torch.tensor(FEATS["dinov3"][0], device=DEVICE)
XD2_T = torch.tensor(FEATS["dinov2"][0], device=DEVICE)
CAT_T = torch.tensor(CAT.astype(np.int64), device=DEVICE)

def fused_features(model, idx, bs=4096):
    model.eval(); out = []
    with torch.no_grad():
        for i in range(0, len(idx), bs):
            j = torch.as_tensor(idx[i:i + bs], device=DEVICE)
            out.append(model(XS_T[j], XD3_T[j], XD2_T[j]).cpu().numpy())
    return np.concatenate(out).astype(np.float32)

def train_fusion(tag, train_species, val_species, out_dir, epochs=80, batch=256, lr=5e-5, wd=1e-2, patience=40):
    out_dir = Path(out_dir); out_dir.mkdir(parents=True, exist_ok=True)
    ck_best, ck_last, log_p = out_dir / "best.pt", out_dir / "latest.pt", out_dir / "log.json"
    tr_idx = np.where(np.isin(CAT, sorted(train_species)))[0]
    va_idx = np.where(np.isin(CAT, sorted(val_species)))[0]
    k_val  = len(np.unique(CAT[va_idx]))
    model = TriFusion(XS_T.shape[1], XD3_T.shape[1], XD2_T.shape[1]).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
    start, best, bad, hist = 0, -1.0, 0, []
    if ck_last.exists():
        ck = torch.load(ck_last, map_location=DEVICE, weights_only=False)
        model.load_state_dict(ck["model"]); opt.load_state_dict(ck["opt"])
        start, best, bad = ck["epoch"] + 1, ck["best"], ck["bad"]
        hist = json.load(open(log_p)) if log_p.exists() else []
        print(f"[{tag}] resuming at epoch {start + 1}, best val NMI {best:.4f}")
    for ep in range(start, epochs):
        if bad >= patience:
            break
        model.train()
        order = np.random.default_rng(SEED + ep).permutation(tr_idx)
        tot, nb = 0.0, 0
        for i in range(0, len(order), batch):
            j = torch.as_tensor(order[i:i + batch], device=DEVICE)
            loss = supcon_loss(model(XS_T[j], XD3_T[j], XD2_T[j]), CAT_T[j])
            opt.zero_grad(); loss.backward(); opt.step()
            tot += float(loss); nb += 1
        f = fused_features(model, va_idx)
        nmi = normalized_mutual_info_score(CAT[va_idx], KMeans(k_val, random_state=SEED, n_init=3).fit_predict(f))
        improved = nmi > best
        if improved:
            best, bad = nmi, 0
            torch.save({"model": model.state_dict(), "best": best}, ck_best)
        else:
            bad += 1
        hist.append({"epoch": ep + 1, "loss": tot / max(nb, 1), "val_nmi": nmi, "best": best})
        json.dump(hist, open(log_p, "w"), indent=1)
        torch.save({"epoch": ep, "best": best, "bad": bad, "model": model.state_dict(), "opt": opt.state_dict()}, ck_last)
        print(f"[{tag}] ep {ep+1:3d}  loss {tot/max(nb,1):.4f}  val NMI {nmi:.4f}" + ("  *best*" if improved else ""))
    model.load_state_dict(torch.load(ck_best, map_location=DEVICE, weights_only=False)["model"])
    print(f"[{tag}] best val NMI {best:.4f}")
    return fused_features(model, np.arange(len(CAT))), model

def pick_val(known_species, n_val, seed=SEED):
    ks = np.array(sorted(known_species))
    ks = np.random.default_rng(seed).permutation(ks)
    n_val = min(n_val, len(ks) // 5)
    return set(ks[n_val:].tolist()), set(ks[:n_val].tolist())

print("fusion ready on", DEVICE, "| rows:", len(CAT))

## Step 12 — Train the fusion on the new split

Trained on known species only, early-stopped on held-back known species. Novel species are never seen,
which the cell asserts before training.

In [ ]:
# CELL: fusion_main
known_sp = {cid for cid, r in ROLE.items() if r == "known"}
novel_sp = {cid for cid, r in ROLE.items() if r in ("novel_known_branch", "novel_unseen_branch")}
train_sp, val_sp = pick_val(known_sp, N_VAL_SPECIES)
assert not (novel_sp & (train_sp | val_sp)), "LEAKAGE: a novel species is in the fusion training data"
print(f"fusion training: {len(train_sp)} train species, {len(val_sp)} validation species, "
      f"{len(novel_sp)} novel species held out")

F_MAIN, _ = train_fusion("main", train_sp, val_sp, V2_DIR / "fusion_main")
np.savez(V2_DIR / "features_trifusion_v2.npz", X=F_MAIN, cat_ids=CAT, relpaths=REL)
json.dump({"train": sorted(train_sp), "val": sorted(val_sp)}, open(V2_DIR / "fusion_main" / "species_folds.json", "w"))
print("saved v2/features_trifusion_v2.npz", F_MAIN.shape)

## Step 13 — Sanity checks on the new benchmark

Quick checks that the new benchmark behaves, comparing the trained fusion with the training-free concatenation.
These are sanity checks only; the full analyses are in `analysis/` (see the README).

- **Router:** cluster-level known-vs-novel AUROC, prototypes from 70% of each known species' images.
- **Cluster validity:** oracle-k NMI on novel species, split by route.


In [ ]:
# CELL: sanity
from sklearn.metrics import roc_auc_score, normalized_mutual_info_score
from sklearn.cluster import KMeans

def l2(a):
    return a / (np.linalg.norm(a, axis=1, keepdims=True) + 1e-8)

def check(X, label):
    rng = np.random.default_rng(SEED)
    roles = np.array([ROLE[c] for c in CAT])
    ref, test = [], []
    for cid in sorted({c for c, r in ROLE.items() if r == "known"}):
        m = np.where(CAT == cid)[0].copy(); rng.shuffle(m)
        k = int(round(0.7 * len(m))); ref += list(m[:k]); test += list(m[k:])
    ref, test = np.array(ref), np.array(test)
    sp = np.unique(CAT[ref])
    protos = l2(np.stack([X[ref[CAT[ref] == c]].mean(0) for c in sp]))
    novel_idx = np.where(np.isin(roles, ["novel_known_branch", "novel_unseen_branch"]))[0]
    def cluster_scores(idx):
        s = 1 - (X[idx] @ protos.T).max(1)
        return np.array([s[CAT[idx] == c].mean() for c in np.unique(CAT[idx])])
    sk, sn = cluster_scores(test), cluster_scores(novel_idx)
    auroc = roc_auc_score(np.r_[np.zeros(len(sk)), np.ones(len(sn))], np.r_[sk, sn])
    res = {"router AUROC (cluster)": auroc}
    for role in ["novel_known_branch", "novel_unseen_branch"]:
        idx = np.where(roles == role)[0]
        if len(np.unique(CAT[idx])) < 2:
            continue
        k = len(np.unique(CAT[idx]))
        pred = KMeans(k, random_state=SEED, n_init=5).fit_predict(X[idx])
        res[f"NMI {role}"] = normalized_mutual_info_score(CAT[idx], pred)
    print(f"{label:<28}" + "  ".join(f"{k} {v:.3f}" for k, v in res.items()))
    return res

concat = l2(np.concatenate([FEATS[n][0] for n in BACKBONES], axis=1))
print("Same benchmark, two representations:\n")
check(concat, "concat (training-free)")
check(F_MAIN, "trained fusion")

## Step 14 — Five-split fusions (while the GPU is free)

The transfer test of the rank radii should run on the trained fusion the system uses, not only on the
training-free concatenation. Here a separate fusion is trained for each hold-out pair, on all pool species
*outside* that pair, so the test runs on the real representation with no leakage. Each run is resumable.
Set `RUN_ALT_SPLITS = False` to skip.


In [ ]:
# CELL: alt_splits
RUN_ALT_SPLITS = True

if RUN_ALT_SPLITS:
    (V2_DIR / "alt_splits").mkdir(exist_ok=True)
    for pair in ALT_PAIRS:
        tag = "+".join(pair).replace(" ", "_")
        out_f = V2_DIR / "alt_splits" / f"features_trifusion_{tag}.npz"
        if out_f.exists():
            print(f"[{tag}] already done"); continue
        known_alt = {s["category_id"] for s in POOL if s["supercategory"] not in pair}
        unseen    = {s["category_id"] for s in POOL if s["supercategory"] in pair}
        tr, va = pick_val(known_alt, N_VAL_SPECIES)
        assert not (unseen & (tr | va))
        F_alt, _ = train_fusion(tag, tr, va, V2_DIR / "alt_splits" / f"fusion_{tag}")
        np.savez(out_f, X=F_alt, cat_ids=CAT, relpaths=REL, unseen=np.array(sorted(unseen)))
        print(f"[{tag}] saved {out_f.name}")

## Step 15 — Manifest: what to copy to the analysis machine

In [ ]:
# CELL: manifest
want = (["species_pool.json", "species_split_v2.json", "alt_splits.json", "items.json",
         "features_siglip2_v2.npz", "features_dinov3_v2.npz", "features_dinov2_v2.npz",
         "features_trifusion_v2.npz"]
        + [f"alt_splits/{p.name}" for p in sorted((V2_DIR / "alt_splits").glob("features_trifusion_*.npz"))]
        + ["fusion_main/best.pt", "fusion_main/log.json", "fusion_main/species_folds.json"])
manifest, total = [], 0
for rel in want:
    p = V2_DIR / rel
    if p.exists():
        size = p.stat().st_size; total += size
        manifest.append({"file": rel, "bytes": size})
        print(f"  {rel:<55}{size/1e6:>9.1f} MB")
    else:
        print(f"  {rel:<55}  MISSING")
summary = {"species": len(POOL), "images": int(len(CAT)) if "CAT" in dir() else None,
           "roles": dict(Counter(ROLE.values())), "seed": SEED, "files": manifest}
json.dump(summary, open(V2_DIR / "MANIFEST.json", "w"), indent=1)
print(f"\ncopy the files above (total {total/1e9:.2f} GB) plus MANIFEST.json to the analysis machine.")
print("v2/feature_shards/ is not needed once the merged files exist; delete it to save space if you like.")

---
### Next

Copy the files listed in the manifest to the analysis machine, point `hnsd_paths` at them (see the README), and run
the analyses in `analysis/`. None of them need a GPU.
